In [3]:
import numpy as np

In [52]:
class CustomInt:
    def __init__(self, value, bit_width):
        """
        Initialize a CustomInt object.

        Args:
            value (int): The initial value.
            bit_width (int): The number of bits for the integer.
        """
        self.bit_width = bit_width
        self.max_value = (1 << (bit_width - 1)) - 1
        self.min_value = -(1 << (bit_width - 1))
        self.value = value
        self._apply_mask()

    def _apply_mask(self):
        """Apply the bit mask to ensure the value fits within the specified bit width."""
        if self.value < 0:
            self.value = (1 << self.bit_width) + self.value
        self.value = self.value & ((1 << self.bit_width) - 1)
        if self.value > self.max_value:
            self.value = self.value - (1 << self.bit_width)

    def __add__(self, other):
        result = self.value + other.value
        return CustomInt(result, self.bit_width)

    def __sub__(self, other):
        result = self.value - other.value
        return CustomInt(result, self.bit_width)

    def __mul__(self, other):
        result = self.value * other.value
        return CustomInt(result, self.bit_width)

    def __floordiv__(self, other):
        result = self.value // other.value
        return CustomInt(result, self.bit_width)

    def __mod__(self, other):
        result = self.value % other.value
        return CustomInt(result, self.bit_width)

    def __and__(self, other):
        result = self.value & other.value
        return CustomInt(result, self.bit_width)

    def __or__(self, other):
        result = self.value | other.value
        return CustomInt(result, self.bit_width)

    def __xor__(self, other):
        result = self.value ^ other.value
        return CustomInt(result, self.bit_width)

    def __lshift__(self, other):
        result = self.value << other
        return CustomInt(result, self.bit_width)

    def __rshift__(self, other):
        result = self.value >> other
        return CustomInt(result, self.bit_width)

    def __repr__(self):
        return f"CustomInt(value={self.value}, bit_width={self.bit_width})"

    def __eq__(self, other):
        return self.value == other.value and self.bit_width == other.bit_width

    def __ne__(self, other):
        return not self.__eq__(other)

    def __lt__(self, other):
        return self.value < other.value

    def __le__(self, other):
        return self.value <= other.value

    def __gt__(self, other):
        return self.value > other.value

    def __ge__(self, other):
        return self.value >= other.value

In [55]:
class FP8:
    def __init__(self, value, exp_bits=4, mant_bits=3):
        """Initialize an FP8 object.

        Args:
            value (int): The 8-bit floating-point value.
            exp_bits (int): Number of bits for the exponent.
            mant_bits (int): Number of bits for the mantissa.
        """

        self.value = value
        self.exp_bits = exp_bits
        self.mant_bits = mant_bits
        self.bias = (1 << (exp_bits - 1)) - 1

        # Calculate the bias, min and max exponent
        if self.exp_bits == 4:
            self.max_exponent = ((1 << self.exp_bits) - 1) - self.bias
        else:
            self.max_exponent = ((1 << self.exp_bits) - 1) - self.bias - 1

        self.min_exponent = 1 - self.bias

        # Print the bias, min and max exponent
        print(f"Bias: {self.bias}, Min Exponent: {self.min_exponent}, Max Exponent: {self.max_exponent}")

    @property
    def precision(self):
        """
        Calculate the precision of the FP8 format.

        Returns:
            float: The precision of the FP8 format.
        """
        return 2 ** (-self.mant_bits)

    @property
    def range(self):
        """
        Calculate the range of the FP8 format.

        Returns:
            tuple: The (min, max) range of the FP8 format.
        """

        # Minimum normal value
        min_norm_value = 2 ** self.min_exponent
        # Maximum normal value
        if self.exp_bits == 4:
            max_norm_value = (2 - self.precision * 2) * 2 ** self.max_exponent
        else:
            max_norm_value = (2 - self.precision) * 2 ** self.max_exponent

        # Print the min and max normal value
        print(f"Min Normal Value: {min_norm_value}, Max Normal Value: {max_norm_value}")
          
        # Minimum subnormal value
        min_subnorm_value = self.precision * 2 ** self.min_exponent
        # Maximum subnormal value
        max_subnorm_value = (1 - self.precision) * 2 ** self.min_exponent

        # Print the min and max subnormal value
        print(f"Min Subnormal Value: {min_subnorm_value}, Max Subnormal Value: {max_subnorm_value}")
        
    def to_fixed_point(self, int_bits=19, frac_bits=18):
        """Convert FP8 value to fixed-point.

        Returns:
            int: The fixed-point representation of the FP8 value.
        """
        sign = (self.value & (1 << (self.exp_bits + self.mant_bits))) >> (self.exp_bits + self.mant_bits)
        exponent = (self.value >> self.mant_bits) & ((1 << self.exp_bits) - 1)
        biased_exponent = exponent - self.bias
        mantissa = self.value & ((1 << self.mant_bits) - 1)
        
        # Print the sign, exponent, and mantissa
        print(f"Sign: {sign}, Exponent: {exponent}, Biased Exponent: {biased_exponent}, Mantissa: {mantissa}")
        
        if exponent == 0:
            fp_value = mantissa * (2 ** self.min_exponent)
        else:
            fp_value = (1 + mantissa * (2 ** (-self.mant_bits))) * (2 ** biased_exponent)
        
        if sign == 1:
            fp_value = -fp_value
            
        # Print the fp_value
        print(f"FP value: {fp_value}")

        # CustomInt object for fixed-point value
        fixed_point_value = CustomInt(int(fp_value * (2 ** frac_bits)), int_bits + frac_bits)

        # fixed_point_value = int(fp_value * (2 ** frac_bits))
        return fixed_point_value
    
    @staticmethod
    def from_fixed_point(fixed_point_value, exp_bits=4, mant_bits=3):
        """Convert fixed-point (Q1.6 format) back to FP8.

        Args:
            fixed_point_value (int): The fixed-point value to convert.
            exp_bits (int): Number of bits for the exponent.
            mant_bits (int): Number of bits for the mantissa.

        Returns:
            FP8: The FP8 representation of the fixed-point value.
        """
        bias = (1 << (exp_bits - 1)) - 1
        fp_value = fixed_point_value / (2 ** 6)
        sign = 0
        if fp_value < 0:
            sign = 1
            fp_value = -fp_value
        
        if fp_value < (2 ** (-bias + 1 - mant_bits)):
            exponent = 0
            mantissa = int(fp_value / (2 ** (-bias + 1 - mant_bits)))
        else:
            exponent = int(np.log2(fp_value)) + bias
            mantissa = int((fp_value / (2 ** (exponent - bias)) - 1) * (2 ** mant_bits))
        
        value = (sign << (exp_bits + mant_bits)) | (exponent << mant_bits) | mantissa
        return FP8(value, exp_bits, mant_bits)

In [57]:
# Test the FP8 class
fp8 = FP8(0b01010101)
# Print the FP8 object
fixed_point_value = fp8.to_fixed_point()
# Print in binary format and decimal format with the dot position
print('Fixed-point value in binary format:')
print('{:037b}'.format(fixed_point_value.value))
print('Fixed-point value in decimal format:')
print(fixed_point_value)
fp8 = FP8.from_fixed_point(fixed_point_value)
# Print in binary format and decimal format
print('FP8 value in binary format:')
print('{:08b}'.format(fp8.value))

Bias: 7, Min Exponent: -6, Max Exponent: 8
Sign: 0, Exponent: 10, Biased Exponent: 3, Mantissa: 5
FP value: 13.0
Fixed-point value in binary format:
00000000000000001101000000000000000000
Fixed-point value in decimal format:
CustomInt(value=3407872, bit_width=37)


TypeError: unsupported operand type(s) for /: 'CustomInt' and 'int'

In [49]:
# Print the range of the FP8 format
fp8.range

Bias: 15, Min Exponent: -14, Max Exponent: 15
Min Normal Value: 6.103515625e-05, Max Normal Value: 57344.0
Min Subnormal Value: 1.52587890625e-05, Max Subnormal Value: 4.57763671875e-05


In [46]:
# Create an FP8 value with a different exponent and mantissa
fp8 = FP8(0b01010101, exp_bits=5, mant_bits=2)
# Print the range of the FP8 format
fp8.range

Bias: 15, Min Exponent: -14, Max Exponent: 15
Min Normal Value: 6.103515625e-05, Max Normal Value: 57344.0
Min Subnormal Value: 1.52587890625e-05, Max Subnormal Value: 4.57763671875e-05
